# MEDISCOPE — 05 Model Evaluation

## Final held-out comparison

This notebook analyses the final evaluation report produced after training.

All four models are compared on the same **60,855-record held-out test set** using a recorded classification threshold of **0.50**.

The evaluation deliberately goes beyond accuracy because LTFU prioritisation requires attention to both:

- **false negatives** — LTFU cases the model fails to flag;
- **false positives** — retained patients incorrectly prioritised as LTFU risk.

## 1. Project setup

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


def find_project_root(start: Path | None = None) -> Path:
    """Locate the MEDISCOPE repository root from common notebook launch locations."""
    start = (start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (
            (candidate / "src").is_dir()
            and (candidate / "api").is_dir()
            and (candidate / "requirements.txt").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Unable to locate the MEDISCOPE repository root. "
        "Run this notebook from the repository or notebooks directory."
    )


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
MODEL_DIR = PROJECT_ROOT / "models" / "trained"
REPORT_DIR = PROJECT_ROOT / "reports" / "evaluation"

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
COMPARISON_FILE = REPORT_DIR / "metrics" / "model_comparison.csv"

if not COMPARISON_FILE.exists():
    raise FileNotFoundError(f"Evaluation comparison not found: {COMPARISON_FILE}")

comparison = pd.read_csv(COMPARISON_FILE)

# Defensive cleanup in case a report export contains Markdown emphasis characters.
comparison.columns = [
    str(column).replace("*", "").strip()
    for column in comparison.columns
]

comparison = comparison.sort_values("descriptive_test_rank").reset_index(drop=True)

comparison

## 2. Headline performance comparison

In [ ]:
headline_columns = [
    "descriptive_test_rank",
    "model",
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall_sensitivity",
    "specificity",
    "f1_score",
    "roc_auc",
    "pr_auc_average_precision",
]

headline = comparison[headline_columns].copy()

percent_columns = [
    column for column in headline_columns
    if column not in {"descriptive_test_rank", "model"}
]

headline[percent_columns] = headline[percent_columns].mul(100)

headline.style.format({
    column: "{:.2f}%"
    for column in percent_columns
})

### Final descriptive ranking

1. **Logistic Regression**
2. **XGBoost**
3. **Random Forest**
4. **AdaBoost**

Logistic Regression achieved the strongest overall held-out performance.

## 3. Compare major classification metrics

In [ ]:
metric_columns = [
    "accuracy",
    "precision",
    "recall_sensitivity",
    "specificity",
    "f1_score",
]

plot_df = (
    comparison
    .set_index("model")[metric_columns]
    .mul(100)
    .T
)

plt.figure(figsize=(11, 6))

x = np.arange(len(plot_df.index))
width = 0.18

for index, model_name in enumerate(plot_df.columns):
    plt.bar(
        x + index * width,
        plot_df[model_name].values,
        width,
        label=model_name,
    )

plt.xticks(
    x + width * (len(plot_df.columns) - 1) / 2,
    plot_df.index,
    rotation=20,
)
plt.ylabel("Score (%)")
plt.title("Held-out classification performance")
plt.legend()
plt.tight_layout()
plt.show()

## 4. Confusion matrices

In [ ]:
confusion_columns = [
    "model",
    "true_negatives",
    "false_positives",
    "false_negatives",
    "true_positives",
]

comparison[confusion_columns]

In [ ]:
figures = []

for _, row in comparison.iterrows():
    matrix = np.array([
        [row["true_negatives"], row["false_positives"]],
        [row["false_negatives"], row["true_positives"]],
    ], dtype=float)

    plt.figure(figsize=(5, 4))
    plt.imshow(matrix)

    for (i, j), value in np.ndenumerate(matrix):
        plt.text(j, i, f"{int(value):,}", ha="center", va="center")

    plt.xticks([0, 1], ["Predicted retained", "Predicted LTFU"])
    plt.yticks([0, 1], ["Actual retained", "Actual LTFU"])
    plt.title(f"Confusion matrix — {row['model']}")
    plt.colorbar(label="Records")
    plt.tight_layout()
    plt.show()

### Clinical interpretation of errors

For the held-out set:

- actual LTFU records: **29,374**
- actual retained records: **31,481**

Logistic Regression produced:

- **28,857 true positives**
- **517 false negatives**
- **31,082 true negatives**
- **399 false positives**

XGBoost produced:

- **28,482 true positives**
- **892 false negatives**
- **30,637 true negatives**
- **844 false positives**

AdaBoost achieved very high sensitivity but generated substantially more false positives, illustrating why sensitivity alone is not a sufficient selection criterion.

## 5. False-positive and false-negative rates

In [ ]:
error_rates = (
    comparison[
        ["model", "false_positive_rate", "false_negative_rate"]
    ]
    .set_index("model")
    .mul(100)
)

error_rates

In [ ]:
error_rates.plot(kind="bar", figsize=(9, 5))
plt.ylabel("Rate (%)")
plt.xlabel("Model")
plt.title("Held-out false-positive and false-negative rates")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 6. Probability-quality metrics

In [ ]:
probability_metrics = comparison[
    [
        "model",
        "roc_auc",
        "pr_auc_average_precision",
        "brier_score",
        "log_loss",
        "matthews_correlation_coefficient",
    ]
]

probability_metrics

ROC-AUC and PR-AUC measure ranking/discrimination, while Brier score and log loss assess probability quality. Matthews correlation coefficient provides a balanced classification summary.

The combination offers a more complete assessment than accuracy alone.

## 7. Prediction throughput and model size

In [ ]:
efficiency = comparison[
    [
        "model",
        "prediction_seconds",
        "predictions_per_second",
        "model_file_size_bytes",
    ]
].copy()

efficiency["model_file_size_mb"] = (
    efficiency["model_file_size_bytes"] / (1024**2)
)

efficiency

The final persisted artefacts differ substantially in size. Logistic Regression is only a few kilobytes, while Random Forest is tens of megabytes. In this project, the compact Logistic Regression model also achieved the strongest final held-out performance.

## 8. Why Logistic Regression and XGBoost were operationalised

In [ ]:
comparison.loc[
    comparison["model"].isin(["Logistic Regression", "XGBoost"]),
    [
        "model",
        "accuracy",
        "precision",
        "recall_sensitivity",
        "specificity",
        "f1_score",
        "roc_auc",
        "false_negatives",
        "false_positives",
    ],
]

MEDISCOPE operationalises **Logistic Regression and XGBoost** rather than collapsing the entire modelling exercise into a single algorithm.

- Logistic Regression provides the strongest final held-out result and a comparatively interpretable baseline.
- XGBoost provides an independently structured, non-linear boosted-tree estimate.
- Showing both outputs allows the clinician interface to expose **agreement and disagreement** instead of pretending that model uncertainty does not exist.

The system therefore treats prediction as decision-support evidence, not as an autonomous clinical verdict.

## Key findings

- Logistic Regression ranked first on the final held-out comparison.
- XGBoost ranked second and remained a strong complementary model.
- Random Forest also performed strongly but was considerably larger.
- AdaBoost demonstrated the trade-off between very high sensitivity and much lower specificity.
- Error counts, probability quality, throughput and model size all contribute to model assessment.
- Final application design preserves separate LR and XGBoost probabilities.

### Next notebook

`06_interpretability.ipynb` moves from performance to interpretation, model agreement and responsible clinical translation.